# Loremaster Embellishment Evaluation

This notebook runs the embellishment eval across multiple prompt variants to measure how much narrative voice the model adds beyond what the retrieved source material contains.

## How It Works

Two-pass scoring using the same `score_narrative` function:

1. Score the retrieved source snippets for narrative quality
2. Score the model response for narrative quality
3. Compute delta — how much narrative the model added on top of the source

Because both scores use the same scale, the delta is directly comparable across questions and variants.

## Metrics

| Metric | Description |
|---|---|
| `source_score` | Narrative quality of retrieved snippets (1=dry, 5=fully narrative) |
| `response_score` | Narrative quality of the model response (1=dry, 5=fully narrative) |
| `delta` | `response_score - source_score` — embellishment added by the model |

A delta near 0 means the model mirrored the source register. A large positive delta means the model added significant narrative voice beyond what the source contained.

## Prompt Variants

| Variant | Instruction |
|---|---|
| `original` | Write a conversational markdown summary that directly answers the question. Use bold and natural prose. |
| `clear_direct` | Write a clear, direct markdown summary that directly answers the question. |
| `match_tone` | Write a markdown summary that directly answers the question. Match the tone of the content provided. |
| `no_adjectives` | Write a markdown summary that directly answers the question. |

## Benchmark

60 questions across three categories: `lore` (20), `gameplay` (20), `item` (20). See `evals/generation/benchmark.json`.

## Retrieval Strategy

Fixed at `rrf_elser` (current production). Fragment size 1500, 3 fragments per document.

In [ ]:
import sys, json
from pathlib import Path
sys.path.append(str(Path('../..').resolve()))

import anthropic
from elasticsearch import Elasticsearch
from config import ANTHROPIC_API_KEY, ES_ENDPOINT, ES_API_KEY, ES_INDEX
from evals.generation.judge import score_narrative
from evals.generation.run_eval import search_with_snippets, generate

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
es     = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY)

BENCHMARK = Path('../../evals/generation/benchmark.json')
STRATEGY  = 'rrf_elser'

PROMPT_VARIANTS = {
    'original':      'Write a conversational markdown summary that directly answers the question. Use bold and natural prose.',
    'clear_direct':  'Write a clear, direct markdown summary that directly answers the question.',
    'match_tone':    'Write a markdown summary that directly answers the question. Match the tone of the content provided.',
    'no_adjectives': 'Write a markdown summary that directly answers the question.',
}

## Load Benchmark

In [ ]:
benchmark  = json.loads(BENCHMARK.read_text())
categories = sorted(set(b['category'] for b in benchmark))

print(f'Total questions: {len(benchmark)}')
for cat in categories:
    print(f'  {cat}: {sum(1 for b in benchmark if b["category"] == cat)}')

## Run Embellishment Eval

For each variant: retrieves snippets from ES, generates a response, scores source and response independently, computes delta.

In [ ]:
all_results = {}

for variant_name, instruction in PROMPT_VARIANTS.items():
    print(f'Running: {variant_name}')
    results = []
    for entry in benchmark:
        snippets   = search_with_snippets(entry['question'], STRATEGY)
        response   = generate(entry['question'], snippets, instruction)
        source_j   = score_narrative('\n---\n'.join(snippets), client)
        response_j = score_narrative(response, client)
        results.append({
            **entry,
            'snippets':           snippets,
            'response':           response,
            'source_score':       source_j['narrative_score'],
            'response_score':     response_j['narrative_score'],
            'delta':              response_j['narrative_score'] - source_j['narrative_score'],
            'source_reasoning':   source_j['reasoning'],
            'response_reasoning': response_j['reasoning'],
        })
    all_results[variant_name] = results
    print(f'  Done ({len(results)} questions scored)')

## Overall Results

In [ ]:
col = 14
print(f"{'':18}", end='')
for name in PROMPT_VARIANTS:
    print(f'  {name:>{col}}', end='')
print()
print('-' * (18 + (col + 2) * len(PROMPT_VARIANTS)))

for field in ['source_score', 'response_score', 'delta']:
    avgs = {n: sum(r[field] for r in all_results[n]) / len(all_results[n]) for n in PROMPT_VARIANTS}
    print(f'{field:<18}', end='')
    for name in PROMPT_VARIANTS:
        print(f'  {avgs[name]:>{col}.2f}', end='')
    print()

## Per-Category Breakdown

In [ ]:
for category in categories:
    print(f'\n{category}')
    for field in ['source_score', 'response_score', 'delta']:
        print(f'  {field:<18}', end='')
        for name in PROMPT_VARIANTS:
            cat_results = [r for r in all_results[name] if r['category'] == category]
            avg = sum(r[field] for r in cat_results) / len(cat_results)
            print(f'  {avg:>12.2f}', end='')
        print()

## High Embellishment Flags

Questions where the model added 2 or more points of narrative beyond the source.

In [ ]:
for name in PROMPT_VARIANTS:
    flagged = [r for r in all_results[name] if r['delta'] >= 2]
    print(f'{name} — {len(flagged)} flagged')
    for r in flagged:
        print(f"  [{r['category']}] {r['question'][:70]}")
        print(f"    source={r['source_score']}  response={r['response_score']}  delta={r['delta']:+d}")
        print(f"    {r['response_reasoning']}")
    print()